In [ ]:
import joblib
import os
import pandas as pd
import numpy as np
import random
import pickle
import hashlib
os.chdir('Resources/')

In [ ]:
model = joblib.load('8_RF_Isolation_Model.joblib')
scaler = joblib.load('5_scaler.joblib')

In [ ]:
def power(a, b, c):
    x = 1
    y = a
    while b > 0:
        if b % 2 != 0:
            x = (x * y) % c
        y = (y * y) % c
        b = int(b / 2)
    return x % c

In [ ]:
def encrypt(value, q, h, g, fixed_key):
    s = power(h, fixed_key, q)
    p = power(g, fixed_key, q)
    return s * value, p

In [ ]:
public_key_file_path = '3_Public_Key.pkl'

with open(public_key_file_path, 'rb') as public_key_file:
    column_keys = pickle.load(public_key_file)

In [ ]:
df_h = pd.read_csv('5_Preprocessed_Data.csv')
df_s = pd.read_csv('1_Structured_Data.csv')

In [ ]:
X = df_s.drop(['HeartDisease'], axis='columns')
Y = df_s[['HeartDisease']]

In [ ]:
count=0
mismatched = []
for i in range(4044):
    row_index = i

    encrypted_row = X.iloc[row_index].copy()

    for column in X.columns:
        q, h, g, fixed_key = column_keys[column]
        value = X.at[row_index, column]
        
        if isinstance(value, str):
            numeric_value = sum(ord(char) for char in value)
            encrypted_value, _ = encrypt(numeric_value, q, h, g, fixed_key)
        else:
            encrypted_value, _ = encrypt(float(value), q, h, g, fixed_key)
        
        encrypted_row[column] = encrypted_value % 999999999

    encrypted_row_df = pd.DataFrame([encrypted_row])

    X_new_scaled_s = scaler.transform(encrypted_row_df)

    pred_new_s = model.predict(X_new_scaled_s)

    X_new_h = df_h.iloc[row_index, :-1]  
    X_new_h = np.array(X_new_h).reshape(1, -1)

    pred_new_h = model.predict(X_new_h)

    if(pred_new_h[0]!=pred_new_s[0]):
        count+=1
        mismatched.append(row_index + 2)
    
    print("Predicted label for", row_index + 2, "th data:", '(', pred_new_h[0], ',', pred_new_s[0], ')', count)

In [11]:
print("No. of mismatched labels:", count)

No. of mismatched labels: 196


In [12]:
print(mismatched)

[1231, 1248, 1253, 1261, 1271, 1273, 1305, 1310, 1313, 1315, 1325, 1328, 1336, 1340, 1346, 1347, 1363, 1366, 1403, 1441, 1454, 1459, 1493, 1515, 1522, 1554, 1567, 1577, 1595, 1608, 1612, 1617, 1636, 1642, 1645, 1667, 1679, 1690, 1691, 1701, 1709, 1710, 1711, 1714, 1721, 1724, 1735, 1746, 1749, 1750, 1761, 1777, 1786, 1789, 1801, 1807, 1853, 1862, 1863, 1897, 1924, 1933, 1934, 1940, 1951, 1957, 1968, 1970, 1974, 1989, 2001, 2002, 2011, 2016, 2017, 2043, 2044, 2053, 2094, 2116, 2119, 2137, 2141, 2159, 2160, 2168, 2169, 2173, 2211, 2214, 2217, 3131, 3133, 3135, 3137, 3139, 3140, 3141, 3146, 3149, 3150, 3156, 3157, 3164, 3165, 3166, 3170, 3171, 3172, 3175, 3179, 3185, 3188, 3189, 3194, 3197, 3200, 3201, 3203, 3204, 3205, 3206, 3208, 3210, 3211, 3218, 3222, 3225, 3231, 3233, 3235, 3240, 3241, 3242, 3249, 3255, 3257, 3263, 3265, 3266, 3270, 3271, 3280, 3281, 3283, 3286, 3289, 3290, 3291, 3293, 3297, 3300, 3303, 3304, 3309, 3310, 3312, 3318, 3383, 3421, 3425, 3450, 3485, 3543, 3545, 3554, 355